NLP


In [5]:
# Install required packages
!pip install requests beautifulsoup4 nltk textblob vaderSentiment \
             pandas matplotlib seaborn wordcloud plotly -q

In [6]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng')


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\snsha\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [11]:
 
# ── Core libraries ──────────────────────────────────────────────
import requests
import json
import re
import time
from datetime import datetime
 
# ── Data handling ────────────────────────────────────────────────
import pandas as pd
 
# ── NLP ──────────────────────────────────────────────────────────
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
 
# ── Web Scraping ─────────────────────────────────────────────────
from bs4 import BeautifulSoup
 
# ── Visualization ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from wordcloud import WordCloud
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
 
print("✅ All imports successful!")
 

✅ All imports successful!


In [26]:
NEWS_API_KEY = '1aeaa7cfb4264aa1a96de57794f8c675'
catagories = ['Healthcare','Technology']
country = 'us'
page_size = 1
search_queries = ['Economy', 'Artificial intelligence']
base_url = 'https://newsapi.org/v2'

top_headline = {
        'catagory':catagories,
        'country':country,
        'page_size':page_size,
}

article_headline = {
        'q':search_queries,
        'pageSize':page_size,
        'language': "en",
        'sortBy': "publishedAt",
}
request_data = [ top_headline, article_headline ]

all_aritcals = []

def fetch_all_articles(data:dict):
        is_catagory = bool(data.get('catagory'))
        url = f"{base_url}/{'top-headlines' if is_catagory else 'everything'}"
        fetching_all = data.get('catagory', []) if is_catagory else data.get('q')

        for i in fetching_all:
                request = data
                if 'catagory' in request:
                        request['catagory'] = i
                else:
                        request['q'] = i
               
        get_data(url,request)


def get_data(url:str,data:dict):

        data['apiKey'] = NEWS_API_KEY

        response = requests.get(url,params=data,timeout=10)

        if(response.status_code != 200):
                print('Getting Error on API call')

        articles = response.json().get('articles',[])
        
        if(articles):
                for art in articles:
                        art['category'] = data.get('catagory', []) if bool(data.get('catagory')) else f"kw:{data.get('q')}"
        all_aritcals.extend(articles)

for cat in request_data:
      fetch_all_articles(cat)


In [29]:
print(json.dumps(all_aritcals[0], indent=2))

def set_data_frame(article:list) -> pd.DataFrame:
    row = []
    for art in all_aritcals:
        row.append({
            'title': art.get('title') or '',
            'description' : art.get('description') or'',
            'source' : art.get('source' ,{}).get('name') or '',
            'url': art.get('url') or '',
            'published_at' : art.get('publishedAt') or '',
            'category' : art.get('category') or '',
            'full_content': " ".join(filter(None,[ art.get('title') or '',  art.get('description') or'',  art.get('content') or'']))

        })
    df = pd.DataFrame(row)
    return df


df= set_data_frame(all_aritcals)
df.head()

{
  "source": {
    "id": "al-jazeera-english",
    "name": "Al Jazeera English"
  },
  "author": "Elizabeth Melimopoulos",
  "title": "NASA releases stunning first photos of Earth from Artemis II moon mission - Al Jazeera",
  "description": "The crew were about 100,000 miles (160,000 kilometres) from Earth and were quickly closing in on the moon.",
  "url": "https://www.aljazeera.com/news/2026/4/3/nasa-releases-first-stunning-artemis-ii-photos-of-earth-from-moon-mission",
  "urlToImage": "https://www.aljazeera.com/wp-content/uploads/2026/04/AP26093499359031-1775242604.jpg?resize=1920%2C1440",
  "publishedAt": "2026-04-04T02:47:49Z",
  "content": "NASA has released the first images taken from inside the Artemis II Orion spacecraft, where four astronauts are currently travelling on a mission around the moon.\r\nIn a photograph shared on Friday, m\u2026 [+3113 chars]",
  "category": "Technology"
}


,title,description,source,url,published_at,category,full_content
0,NASA releases stunning first photos of Earth f...,"The crew were about 100,000 miles (160,000 kil...",Al Jazeera English,https://www.aljazeera.com/news/2026/4/3/nasa-r...,2026-04-04T02:47:49Z,Technology,NASA releases stunning first photos of Earth f...
1,Anthropic Cuts Off OpenClaw Support for Claude...,Anthropic said third-party tools like OpenClaw...,Business Insider,https://www.businessinsider.com/anthropic-cuts...,2026-04-04T02:25:00Z,Technology,Anthropic Cuts Off OpenClaw Support for Claude...
2,"Geno Auriemma, Dawn Staley have heated exchang...",Two of women’s basketball’s biggest names had ...,NBC News,https://www.nbcnews.com/sports/college-basketb...,2026-04-04T02:10:00Z,Technology,"Geno Auriemma, Dawn Staley have heated exchang..."
3,Iran and US race to find missing American pilo...,Iranian security forces are searching for the ...,BBC News,https://www.bbc.com/news/live/cm29zmpdj3vt,2026-04-04T02:00:44Z,Technology,Iran and US race to find missing American pilo...
4,White House budget proposal silent on civilian...,The White House proposed up to a 7% pay raise ...,Federal News Network,https://federalnewsnetwork.com/budget/2026/04/...,2026-04-04T00:58:37Z,Technology,White House budget proposal silent on civilian...
